# Day 07 · 에이전트 직접 구성

**에이전트 = LLM + 도구 + 루프.** 5강에서 만든 `run_agent`를 제대로 된 에이전트로 키운다 — ReAct 루프·종료조건·메모리·self-correction을 붙이고, 내 MCP 도구를 물린 뒤, Claude Code/Codex(프로덕션 하네스)와 비교한다.

- LLM = **NVIDIA API(Qwen)**
- 도구 백엔드 = 이 노트북 안에서 정의한 작은 MCP 서버 (5강 예제/배포 URL로 교체 가능)
- **에이전트 ≠ 하네스**: 오늘 만드는 건 판단 주체(에이전트). 그걸 돌리는 런타임(하네스)은 8강.

## Lab 0 · 셋업

패키지 설치 → 도구 서버 정의 → NVIDIA LLM 연결.

In [ ]:
%pip install -q fastmcp openai

In [ ]:
# 이 노트북은 '도구 백엔드'로 쓸 작은 MCP 서버를 여기서 직접 정의한다 (자기완결).
# 5강 예제 server.py를 그대로 써도 되고, 배포된 URL에 붙여도 된다 (Lab 5 참고).
from fastmcp import FastMCP, Client

mcp = FastMCP("Day07AgentTools")

DOCS = [
    {"id": "vacation", "text": "연차는 사용 3영업일 전에 사내포털에서 신청한다."},
    {"id": "remote",   "text": "재택근무는 주 2회까지 가능하며 전날 18시까지 공유한다."},
    {"id": "expense",  "text": "비용 정산은 영수증을 첨부해 정산 메뉴에서 접수한다."},
]
TASKS = []
LEVELS = {"낮음", "보통", "높음"}

@mcp.tool
def search_docs(query: str, k: int = 2) -> list:
    """질문과 관련된 사내 문서를 찾는다 (읽기)"""
    scored = sorted(DOCS, key=lambda d: -sum(w in d["text"] for w in query.split()))
    return scored[:k]

@mcp.tool
def add_task(title: str) -> dict:
    """할 일을 추가한다 (부작용)"""
    TASKS.append({"id": len(TASKS) + 1, "title": title, "level": "보통"})
    return TASKS[-1]

@mcp.tool
def set_priority(task_id: int, level: str) -> dict:
    """작업 우선순위를 바꾼다. level은 낮음·보통·높음 중 하나 (아니면 친절한 에러)."""
    if level not in LEVELS:
        raise ValueError(f"level은 {sorted(LEVELS)} 중 하나여야 합니다. 받은 값: '{level}'")
    for t in TASKS:
        if t["id"] == task_id:
            t["level"] = level
            return t
    raise ValueError(f"{task_id}번 작업이 없습니다.")

@mcp.tool
def list_tasks() -> list:
    """할 일 목록을 읽는다 (읽기)"""
    return TASKS

print("도구 서버 준비됨: search_docs · add_task · set_priority · list_tasks")

In [ ]:
# NVIDIA API로 Qwen(LLM)을 부른다. 토큰은 입력창/환경변수로 (하드코딩 금지).
import os, getpass, json
from openai import OpenAI

LLM_BASE_URL = os.getenv("QWEN_BASE_URL", "https://integrate.api.nvidia.com/v1")
NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY") or getpass.getpass("NVIDIA API 토큰(nvapi-...) 입력: ")
llm = OpenAI(base_url=LLM_BASE_URL, api_key=NVIDIA_API_KEY)

_av = [m.id for m in llm.models.list().data]
LLM_MODEL = os.getenv("LLM_MODEL", "qwen/qwen3-next-80b-a3b-instruct")
if LLM_MODEL not in _av:                     # thinking 모델은 제외 (토큰 폭주 방지)
    _q = [m for m in _av if m.startswith("qwen/") and not any(x in m for x in ("vl", "embed", "rerank", "thinking"))]
    LLM_MODEL = _q[0] if _q else _av[0]
print("모델:", LLM_MODEL)

## Lab 1 · 최소 에이전트 해부

에이전트 루프의 뼈대: **① LLM 판단 → ② 도구 실행 → ③ 결과 되먹임 → 반복**. 도구를 더 안 부르면 정상 종료, `max_steps`를 넘으면 안전 종료.

In [ ]:
# ① MCP 도구(name·description·inputSchema) → OpenAI function 스펙으로 변환
def to_openai_tools(mcp_tools):
    return [{"type": "function", "function": {
        "name": t.name, "description": t.description or "", "parameters": t.inputSchema}} for t in mcp_tools]

# ② 최소 에이전트 루프 — 판단 → 도구 실행 → 되먹임 → 반복
async def run_agent(question, max_steps=5):
    async with Client(mcp) as client:                       # 도구 백엔드 = 내 MCP 서버
        tools = to_openai_tools(await client.list_tools())
        messages = [{"role": "system", "content": "너는 업무 비서다. 필요하면 도구로 처리한 뒤 한국어로 간결히 답하라."},
                    {"role": "user", "content": question}]
        for step in range(max_steps):                       # 반복 한도 = 안전장치
            m = llm.chat.completions.create(
                model=LLM_MODEL, messages=messages, tools=tools, temperature=0.2, max_tokens=400,
                extra_body={"chat_template_kwargs": {"enable_thinking": False}},
            ).choices[0].message
            if not m.tool_calls:                            # 도구를 안 부르면 → 정상 종료
                return m.content
            messages.append({"role": "assistant", "content": m.content or "",
                             "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
            for tc in m.tool_calls:                         # LLM이 고른 도구를 실제 실행
                args = json.loads(tc.function.arguments)
                print(f"  [LLM → 도구] {tc.function.name}({args})")
                res = await client.call_tool(tc.function.name, args)
                messages.append({"role": "tool", "tool_call_id": tc.id,
                                 "content": json.dumps(res.data, ensure_ascii=False, default=str)})
        return "(반복 한도 초과)"                            # 안전 종료

In [ ]:
# 한 번의 질문 → 에이전트가 스스로 search_docs 를 부르고, 그 근거로 답한다
print("Q: 연차는 며칠 전에 신청해?")
print("A:", await run_agent("연차는 며칠 전에 신청해?"))

## Lab 2 · 루프 승격 — 멀티스텝·종료조건

한 질문에 도구가 여러 번 필요하면 루프가 여러 번 돈다. `[LLM → 도구]` 로그가 몇 번 찍히는지 보라. `max_steps`(반복 한도)가 무한 루프·비용 폭주를 막는 안전장치다.

In [ ]:
# 멀티스텝 — 검색과 행동을 이어서 (루프가 여러 번 돈다)
print("Q: '분기 보고서 작성' 할 일을 추가하고, 지금 할 일 목록을 보여줘")
print("A:", await run_agent("'분기 보고서 작성' 할 일을 추가하고, 지금 할 일 목록을 보여줘"))

## Lab 3 · 메모리 — 대화가 곧 기억

`run_agent`는 매 호출이 새 대화라 이전을 기억하지 못한다. `messages`를 여러 질문에 걸쳐 유지하는 `Agent`를 만들면 "방금 그 작업" 같은 지시를 이해한다.

> 대화가 길어지면 요약·트리밍이 필요해진다 → 8강 컨텍스트 엔지니어링.

In [ ]:
# 메모리 = 대화(messages)를 여러 질문에 걸쳐 유지하는 에이전트
class Agent:
    def __init__(self, system):
        self.system = system
        self.messages = [{"role": "system", "content": system}]

    async def ask(self, question, max_steps=5):
        async with Client(mcp) as client:
            tools = to_openai_tools(await client.list_tools())
            self.messages.append({"role": "user", "content": question})   # 이전 대화 위에 쌓인다
            for step in range(max_steps):
                m = llm.chat.completions.create(
                    model=LLM_MODEL, messages=self.messages, tools=tools, temperature=0.2, max_tokens=400,
                    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
                ).choices[0].message
                if not m.tool_calls:
                    self.messages.append({"role": "assistant", "content": m.content})
                    return m.content
                self.messages.append({"role": "assistant", "content": m.content or "",
                                      "tool_calls": [tc.model_dump() for tc in m.tool_calls]})
                for tc in m.tool_calls:
                    args = json.loads(tc.function.arguments)
                    print(f"  [LLM → 도구] {tc.function.name}({args})")
                    try:
                        res = await client.call_tool(tc.function.name, args)
                        content = json.dumps(res.data, ensure_ascii=False, default=str)
                    except Exception as e:              # 에러도 관찰로 되먹임 → 스스로 수정
                        content = f"오류: {str(e).splitlines()[-1]}"
                        print(f"       ↳ 오류를 LLM에 전달: {content}")
                    self.messages.append({"role": "tool", "tool_call_id": tc.id, "content": content})
            return "(반복 한도 초과)"

In [ ]:
# 메모리 확인 — 두 번째 질문의 '방금 그 작업'은 이전 대화를 기억해야 풀린다
agent = Agent("너는 업무 비서다. 도구로 처리하고 한국어로 간결히 답하라.")
print("Q1:", "'디자인 리뷰' 할 일을 추가해줘")
print("A1:", await agent.ask("'디자인 리뷰' 할 일을 추가해줘"))
print("\nQ2:", "방금 추가한 그 작업을 우선순위 높음으로 바꿔줘")
print("A2:", await agent.ask("방금 추가한 그 작업을 우선순위 높음으로 바꿔줘"))

## Lab 4 · self-correction — 에러를 되먹여 스스로 고친다

도구가 던진 에러를 **관찰처럼 되먹이면** LLM이 인자를 고쳐 다시 시도한다. 그래서 5강에서 배운 **친절한 에러**가 중요하다 — 뭘 어떻게 고칠지 알려줘야 한다.

In [ ]:
# self-correction — '긴급'은 허용값(낮음·보통·높음)이 아니라 set_priority 가 친절한 에러를 던진다.
# 에이전트는 그 에러를 읽고 '높음'으로 고쳐 다시 부른다 ([LLM → 도구] 로그가 두 번 찍힌다).
agent2 = Agent("너는 업무 비서다. 도구로 처리하고 한국어로 간결히 답하라.")
print("Q:", "'발표 준비' 할 일을 추가하고, 그 작업을 '긴급'으로 설정해줘")
print("A:", await agent2.ask("'발표 준비' 할 일을 추가하고, 그 작업을 '긴급'으로 설정해줘"))

## Lab 5 · 도구 백엔드 = 진짜 MCP 서버

'에이전트 루프'와 '도구가 어디 있는지'는 분리돼 있다. `Client(mcp)`(인메모리)를 `Client("http://...")`(원격/배포)로 바꿔도 루프는 그대로다.

In [ ]:
# Lab 5 — 도구 백엔드는 '진짜 MCP 서버'다. 인메모리(Client(mcp)) 대신
# 배포된 서버 URL이나 로컬 HTTP로 바꿔도 run_agent 는 그대로 동작한다.
#
# 예) 5강 deploy 서버를 로컬 HTTP로 띄운 뒤:
#     터미널> fastmcp run day05/deploy/server.py --transport http --port 8000
#
# async def run_agent_remote(question, url="http://localhost:8000/mcp", max_steps=5):
#     async with Client(url) as client:          # ← Client(mcp) 를 Client(url) 로만 바꿈
#         tools = to_openai_tools(await client.list_tools())
#         ...                                      # 나머지 루프는 동일
#
# 즉 '에이전트 루프'와 '도구가 어디 있는지'는 분리돼 있다.
print("도구 백엔드는 Client(mcp)=인메모리 ↔ Client(url)=원격 으로 바꿔 끼울 수 있다.")

## Lab 6 · 프로덕션과 비교 — Claude Code / Codex

내 루프에 시킨 것과 **같은 과제**를 Claude Code(또는 Codex)에도 시켜보고, 무엇이 다른지 관찰·기록한다. (코드가 아니라 관찰 실습)

| 관찰 항목 | 내 에이전트 (오늘) | Claude Code / Codex |
|---|---|---|
| 계획 세우기 | 없음(한 번에) | ? |
| 도구 선택 | 내가 준 4개 | ? |
| 컨텍스트 관리 | messages 그대로 | ? (자동 요약/컴팩션) |
| 권한·확인 | 없음 | ? |
| 서브에이전트 | 없음 | ? |

→ 빈칸을 채우며 관찰한 "런타임이 대신 해준 것"이 **8강 하네스**의 출발점이다.

**여기서부터 여러분은 Claude Code/Codex로 작업한다** — 에이전트로 에이전트를 만드는 셈.

## 산출물 체크리스트

- [ ] 최소 에이전트 루프를 이해하고 돌렸다 (`run_agent`)
- [ ] 멀티스텝·종료조건·안전장치를 확인했다
- [ ] 메모리를 유지하는 `Agent`로 이어지는 지시를 처리했다
- [ ] self-correction(에러 되먹임)으로 스스로 고치는 걸 봤다
- [ ] 도구 백엔드가 MCP 서버임을 확인했다 (URL 교체 가능)
- [ ] 같은 과제를 Claude Code/Codex에 시키고 비교 노트를 남겼다